In [ ]:
from google.colab import files
from PIL import Image

uploaded = files.upload()  # Choose nightlights_flat_change_still_8k.00700.jpg

image_path = next(iter(uploaded))
image = Image.open(image_path).convert("RGB")

print("File:", image_path)
print("Size:", image.size)
display(image)

In [ ]:
## World Map Visual Sonification

'''
This code uses NASA’s black-background world map image to create a short synchronized audio and video preview.

- Image processing: Resizes the image to 768 × 432 pixels and classifies visible purple and orange pixels using brightness, saturation, and hue thresholds.
- Heatmap: Recolors the classified pixels while preserving their brightness. Unclassified pixels appear black.
- Sonification: Scans the map from left to right in 48 vertical sections. Each section produces a note:
  - Purple pixels shift the pitch higher.
  - Orange pixels shift the pitch lower.
  - The number of classified pixels affects the volume.
- Visualization: Adds a continuous vertical bar that sweeps left to right over 12 seconds.
- Output: Saves the audio as `world_scan.wav` and combines it with the scanning heatmap video as `world_scan_continuous.mp4`.

This is an exploratory sonification of the colors in the rendered NASA image. The color thresholds describe the visual representation; they do not directly measure the underlying nighttime-light data.
'''

In [ ]:
import shutil
import subprocess
import numpy as np
from PIL import Image, ImageDraw
from scipy.io.wavfile import write as write_wav
from matplotlib.colors import rgb_to_hsv, hsv_to_rgb
from IPython.display import Video, display

# `image` should be the black-background NASA image already loaded
width, height = 768, 432
small = image.convert("RGB").resize((width, height), Image.Resampling.BOX)

# Classify purple and orange pixels using the current thresholds
rgb = np.asarray(small).astype(np.float32) / 255.0
hsv = rgb_to_hsv(rgb)
hue = hsv[:, :, 0] * 360
saturation = hsv[:, :, 1]
brightness = hsv[:, :, 2]

visible_color = (brightness > 0.20) & (saturation > 0.18)
is_orange = visible_color & (hue >= 10) & (hue <= 65)
is_purple = visible_color & (hue >= 235) & (hue <= 320)

# Build a heatmap-style display, preserving the image's brightness
display_hsv = np.zeros_like(hsv)
display_hsv[:, :, 2] = brightness
display_hsv[is_orange, 0] = 30 / 360
display_hsv[is_purple, 0] = 280 / 360
display_hsv[is_orange | is_purple, 1] = 0.75

heatmap = Image.fromarray(
    (hsv_to_rgb(display_hsv) * 255).astype(np.uint8)
)

# Generate the original detailed audio: 48 notes over 12 seconds
scan_sections = 48
seconds = 12
sample_rate = 22050
samples_total = seconds * sample_rate
audio = np.zeros(samples_total, dtype=np.float32)

for section in range(scan_sections):
    x0 = round(section * width / scan_sections)
    x1 = round((section + 1) * width / scan_sections)

    orange_count = int(is_orange[:, x0:x1].sum())
    purple_count = int(is_purple[:, x0:x1].sum())
    total_count = orange_count + purple_count

    if total_count == 0:
        continue

    # Purple shifts pitch up; orange shifts pitch down
    balance = (purple_count - orange_count) / total_count
    frequency = 330 * (2 ** (0.5 * balance))

    # More classified pixels make the note louder
    density = total_count / (height * max(1, x1 - x0))
    gain = 0.22 * np.sqrt(density)

    start = round(section * samples_total / scan_sections)
    end = round((section + 1) * samples_total / scan_sections)
    n = end - start
    t = np.arange(n) / sample_rate

    # Short fades help prevent clicks at note boundaries
    envelope = np.minimum(t / 0.015, 1.0)
    envelope *= np.minimum((n / sample_rate - t) / 0.025, 1.0)

    audio[start:end] = gain * envelope * np.sin(2 * np.pi * frequency * t)

# Save the WAV audio
peak = np.max(np.abs(audio))
if peak > 0:
    audio *= 0.8 / peak

wav_path = "world_scan.wav"
write_wav(wav_path, sample_rate, (audio * 32767).astype(np.int16))

# Render a continuous left-to-right scan bar
fps = 30
frame_count = seconds * fps
base = heatmap.convert("RGB")

ffmpeg = shutil.which("ffmpeg")
if ffmpeg is None:
    raise RuntimeError("Colab could not find ffmpeg.")

video_path = "world_scan_continuous.mp4"
command = [
    ffmpeg, "-y",
    "-f", "rawvideo", "-vcodec", "rawvideo",
    "-pix_fmt", "rgb24", "-s", f"{width}x{height}",
    "-r", str(fps), "-i", "-",
    "-i", wav_path,
    "-c:v", "libx264", "-preset", "ultrafast",
    "-pix_fmt", "yuv420p",
    "-c:a", "aac", "-shortest",
    video_path,
]

process = subprocess.Popen(command, stdin=subprocess.PIPE)

for frame_number in range(frame_count):
    progress = frame_number / (frame_count - 1)
    x = round(progress * (width - 1))

    frame = base.copy()
    draw = ImageDraw.Draw(frame)
    draw.line((x, 0, x, height), fill=(255, 255, 255), width=3)
    process.stdin.write(np.asarray(frame).tobytes())

process.stdin.close()
return_code = process.wait()

if return_code != 0:
    raise RuntimeError("Video creation failed. Check the ffmpeg output above.")

print("Created:", video_path)
display(Video(video_path, embed=True, width=768))